# Fine-tune a local clinical note model — Green Zone EDHKL

This notebook fine-tunes a small Llama model with **LoRA** (via [Unsloth](https://unsloth.ai)) on your
collected `training_examples.jsonl` dataset, so it drafts MOH ED clerking notes in your house style
more reliably than prompting alone. It exports a **GGUF** file at the end that you download and load
into **Ollama** on your own laptop — training happens here (free Colab GPU), inference stays local.

**Before running:** Runtime → Change runtime type → **T4 GPU**.

**You'll need two files from your `V2textDoc` project** (upload when prompted below):
- `backend/data/training_examples.jsonl` — your collected examples
- `backend/clinical_note_prompt.py` — the exact system prompt used at inference, so training matches
  what the app actually sends the model at runtime.


## 1. Install dependencies

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # local/non-Colab setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2


## 2. Upload your dataset + system prompt

Upload **both** `training_examples.jsonl` and `clinical_note_prompt.py` from your `V2textDoc`
project when the file picker appears.


In [ ]:
from google.colab import files
uploaded = files.upload()
assert "training_examples.jsonl" in uploaded, "Missing training_examples.jsonl"
assert "clinical_note_prompt.py" in uploaded, "Missing clinical_note_prompt.py"
print("Got:", list(uploaded.keys()))


## 3. Build the training conversations (matches the app's real inference format)

In [ ]:
import json, sys
sys.path.insert(0, ".")
from clinical_note_prompt import MOH_ED_SYSTEM_PROMPT, build_user_message

raw_examples = []
with open("training_examples.jsonl", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            raw_examples.append(json.loads(line))

print(f"Loaded {len(raw_examples)} training examples")

conversations = []
for rec in raw_examples:
    user_msg = build_user_message(rec["transcript"], rec.get("additional_docs", ""))
    conversations.append({
        "conversations": [
            {"role": "system", "content": MOH_ED_SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": rec["final_note"].strip()},
        ]
    })

print("Example conversation (first case):")
print(conversations[0]["conversations"][1]["content"][:300], "...")


## 4. Load the base model

Uses Meta's Llama 3.1 8B Instruct in 4-bit (matches `llama3.1:8b` you've already been running via
Ollama, so the fine-tune builds on a model you've already evaluated). Swap `model_name` below if you
want to fine-tune a smaller size instead (e.g. `unsloth/Llama-3.2-3B-Instruct-bnb-4bit`).


In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None          # auto-detect
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)


## 5. Apply Llama 3.1's chat template and format the dataset

In [ ]:
from unsloth.chat_templates import get_chat_template
from datasets import Dataset

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1",
)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        for convo in convos
    ]
    return {"text": texts}

dataset = Dataset.from_list(conversations)
dataset = dataset.map(formatting_prompts_func, batched=True)

print(dataset[0]["text"][:500], "...")


## 6. Add LoRA adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)


## 7. Train

With a small dataset (a few dozen examples), epochs matter more than steps — this trains for
several passes over your data rather than a fixed step count.

**Real result from a 15-example / 5-epoch run**: the model got noticeably *worse* on several
cases, not better — it picked up a specific phrase pattern from the 2-3 sepsis-related training
examples ("lactate/blood cultures not mentioned") and started inserting it into completely
unrelated cases (a STEMI, an asthma exacerbation, a head injury) where it made no clinical sense.
That's overfitting: the model memorized surface phrasing from too few examples instead of learning
when the reasoning actually applies. It also got *less* consistent at flagging ambiguous
transcription artifacts, a behavior the prompt alone had already improved.

Default `num_train_epochs` below is set to 2 to reduce this risk. Don't increase it until the
dataset is meaningfully bigger (50+ examples covering a wide range of presentations) — more epochs
on a small dataset amplifies overfitting, it doesn't fix under-training. If you rerun with more
data and the model still isn't following the format well, that's when to consider raising epochs.


In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = False,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 2,     # keep low until the dataset is much bigger - see overfitting note above
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

# Only compute loss on the assistant's note (not the system/user prompt tokens),
# so training focuses on what the model should learn to generate.
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

trainer_stats = trainer.train()


## 8. Quick sanity check

Try it on a transcript style it hasn't seen verbatim, and confirm it (a) doesn't refuse, and
(b) follows the section headings.


In [ ]:
FastLanguageModel.for_inference(model)

test_transcript = (
    "Encik Zul, 45 year old male, known asthmatic, presented with sudden onset "
    "severe shortness of breath after being stung by a bee 20 minutes ago, with "
    "facial swelling and generalised hives. No known drug allergy. On examination "
    "patient anxious, stridor present, blood pressure 82 over 50, heart rate 130, "
    "saturation 91% on room air. Given IM adrenaline, impression anaphylaxis, for admission."
)
messages = [
    {"role": "system", "content": MOH_ED_SYSTEM_PROMPT},
    {"role": "user", "content": build_user_message(test_transcript, "")},
]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

from transformers import TextStreamer
streamer = TextStreamer(tokenizer, skip_prompt=True)
_ = model.generate(input_ids=inputs, streamer=streamer, max_new_tokens=800, use_cache=True)


## 9. Save the LoRA adapter (small, fast — no GGUF conversion needed)

Earlier versions of this notebook merged the LoRA into the full 8B model and converted the whole
thing to GGUF here — that step repeatedly hung/timed out/ran out of disk in testing, because it
means building llama.cpp from source and processing a ~16GB merged model inside the Colab VM.

Ollama can load a LoRA adapter directly on top of a base model it already has, via the Modelfile
`ADAPTER` instruction — no merging, no GGUF conversion, no llama.cpp. This just saves the small
adapter weights (tens of MB, not gigabytes), which is fast and reliable.


In [ ]:
model.save_pretrained("lora_adapters")
tokenizer.save_pretrained("lora_adapters")

!ls -la lora_adapters


## 10. Save the result to Google Drive

The adapter folder is small (tens of MB), so a direct `files.download()` would likely work too —
but routing through Drive is more reliable and consistent with the rest of this notebook.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
dest_dir = "/content/drive/MyDrive/edhkl_clinical_notes"
if os.path.exists(dest_dir):
    shutil.rmtree(dest_dir)
shutil.copytree("lora_adapters", dest_dir)

print("Done. Check drive.google.com -> 'edhkl_clinical_notes' folder in My Drive.")
!ls -la {dest_dir}


## 11. Load it into Ollama on your laptop

You already have `llama3.1:8b` pulled in Ollama — this adapter applies on top of it, so there's no
new base model to download.

1. Download the `edhkl_clinical_notes` folder from `drive.google.com` (My Drive) — or if you have
   Google Drive Desktop syncing, it's already local. It should contain `adapter_model.safetensors`,
   `adapter_config.json`, and tokenizer files.
2. Put it inside your `V2textDoc` project, e.g. `V2textDoc\models\edhkl_clinical_notes\`.
3. In that same `V2textDoc\models\` folder, create a plain text file named `Modelfile` (no
   extension) containing exactly:
   ```
   FROM llama3.1:8b
   ADAPTER ./edhkl_clinical_notes
   ```
4. Open a terminal in `V2textDoc\models\` and run:
   ```
   ollama create edhkl-clinical-notes -f Modelfile
   ```
5. In `backend\.env`, set:
   ```
   OLLAMA_MODEL=edhkl-clinical-notes
   ```
6. `docker compose down` then `docker compose up` (no rebuild needed, only `.env` changed).
7. Try Generate Clinical Note again on `localhost:8000` and compare against the stock
   `llama3.1:8b` output.

If it performs worse or acts strangely (e.g. always returns near-identical text regardless of
input — a sign of overfitting on too few examples), that's a signal to collect more training
examples and re-run this notebook. If it starts inserting irrelevant flags/phrases that showed up in only a couple of training examples (like the lactate/blood-cultures issue above), that's the same overfitting pattern - add more diverse examples rather than increasing epochs.

**If the Colab runtime disconnects/recycles before you reach this step**, everything in `/content`
is wiped and training must be re-run from the top. This should now be much less likely to matter,
though, since steps 8-10 (sanity check → save adapter → copy to Drive) are fast — the slow,
disconnect-prone GGUF/llama.cpp step is gone entirely.
